# Spike train smoothing

Turning spike times into a firing rate means smoothing a point process. nelpy has three pipelines for it, all built on a Gaussian kernel. They differ in where spikes get rounded to a grid, how they treat the edges of the support and the gaps between support intervals, what they return, and how the kernel width $\sigma$ is chosen.

| Pipeline | Spikes enter as | Returns | At support edges | Across support gaps | $\sigma$ |
| --- | --- | --- | --- | --- | --- |
| `st.bin(ds).smooth(sigma)` | counts per `ds` bin | smoothed counts per bin | reflected | smoothed across by default | one value for all units |
| `st.bin(fine).smooth(sigma).rebin(w)` | counts per fine bin | smoothed counts per coarse bin | reflected | smoothed across by default | one value for all units |
| `nel.utils.kernel_rate(st, ds=..., sigma=...)` | exact spike times | rate in Hz at bin centers | renormalized | never | one value per unit, e.g. from `nel.utils.kernel_bandwidth` |

`nel.utils.locsmooth` and `nel.utils.savgol_filter` are not on this list: on binned spike counts they can return negative firing rates (see the *Local Signal Smoothing* tutorial).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import ndtr

import nelpy as nel

%matplotlib inline

## 1. Binning before smoothing loses spike timing

Nine spikes in one second, counted in 200 ms bins and smoothed with $\sigma$ = 100 ms. Five spikes sit at the right edge of the third bin and one at the left edge of the fourth, so the smoothed count should be much larger in bin 4 than in bin 2.

As a reference, the exact smoothed count in a bin $[a, b)$ is the integral of the Gaussian-smoothed spike train over that bin, which has a closed form: $\sum_i \Phi\left(\frac{b - t_i}{\sigma}\right) - \Phi\left(\frac{a - t_i}{\sigma}\right)$, where $\Phi$ is the standard normal CDF.

In [ ]:
def exact_counts(spike_times, edges, sigma):
    """Integral of the Gaussian-smoothed spike train over each bin."""
    upper = ndtr((edges[1:, None] - spike_times[None, :]) / sigma)
    lower = ndtr((edges[:-1, None] - spike_times[None, :]) / sigma)
    return (upper - lower).sum(axis=1)


spikes = np.array([0.1, 0.3, 0.55, 0.56, 0.57, 0.58, 0.59, 0.62, 0.9])
st = nel.SpikeTrainArray(spikes, support=nel.EpochArray([0, 1]), fs=1000)
ds, sigma = 0.2, 0.1
edges = np.linspace(0, 1, 6)

counts = {
    "exact": exact_counts(spikes, edges, sigma),
    "bin 200 ms -> smooth": st.bin(ds=ds).smooth(sigma=sigma).data[0],
    "bin 10 ms -> smooth -> rebin(w=20)": st.bin(ds=0.01)
    .smooth(sigma=sigma)
    .rebin(w=20)
    .data[0],
    "kernel_rate x ds": nel.utils.kernel_rate(st, ds=ds, sigma=sigma).data[0] * ds,
}
for name, values in counts.items():
    print(f"{name:>36}: {np.round(values, 2)}")

In [ ]:
fine_rate = nel.utils.kernel_rate(st, ds=0.001, sigma=sigma, edge_correction=False)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
top.plot(
    fine_rate.abscissa_vals, fine_rate.data[0], color="k", label="smoothed spike train"
)
top.plot(spikes, np.zeros_like(spikes), "|", color="C3", ms=20, mew=2, label="spikes")
for edge in edges:
    top.axvline(edge, color="0.85", zorder=-1)
top.set(ylabel="rate (Hz)")
top.legend(frameon=False, loc="upper left")

centers = edges[:-1] + ds / 2
width = ds / (len(counts) + 1)
for i, (name, values) in enumerate(counts.items()):
    bottom.bar(centers + (i - 1.5) * width, values, width=width, label=name)
bottom.set(xlabel="time (s)", ylabel="smoothed spikes per bin")
bottom.legend(frameon=False, fontsize="small", loc="upper left")
plt.tight_layout()
plt.show()

Smoothing the 200 ms counts moves the five clustered spikes to the middle of bin 3 and spreads them equally into bins 2 and 4 (1.43 each), whereas the exact counts are 1.08 and 2.56. Binning at 10 ms first and summing back to 200 ms recovers the interior bins to within about 2%.

`kernel_rate` uses the exact spike times but returns the rate *at* each bin center, not the average over the bin. Multiplied by `ds` it is about 5% off here, because the bins are wider than $\sigma$; it matches the exact counts when `ds` is small compared with $\sigma$.

The edge bins differ by design (section 3): the exact reference loses the kernel mass that falls outside $[0, 1]$, `smooth` reflects it back in, and `kernel_rate` rescales for it.

## 2. How fine do the fine bins need to be?

`rebin(w)` sums `w` consecutive bins, so `w` must be a whole number: pick a fine bin width that divides the target bin. For 125 ms bins that means 62.5, 25, 5 or 1 ms, but not 50 ms, which would need `w = 2.5`.

The cell below simulates two minutes of spikes from a known rate and compares each pipeline's counts per 125 ms bin with the exact counts, for a kernel wider than the bin ($\sigma$ = 300 ms) and one narrower than it ($\sigma$ = 50 ms).

In [ ]:
def true_rate(t):
    return 3 + 20 * np.exp(-0.5 * (((t % 20) - 10) / 1.0) ** 2)


rng = np.random.default_rng(0)
duration, peak = 120, 23
proposals = np.sort(rng.uniform(0, duration, rng.poisson(peak * duration)))
sim_spikes = proposals[rng.uniform(0, peak, proposals.size) < true_rate(proposals)]
sim_st = nel.SpikeTrainArray(
    sim_spikes, support=nel.EpochArray([0, duration]), fs=30000
)

target = 0.125
edges = np.arange(0, duration + target / 2, target)
interior = (edges[:-1] > 2) & (edges[1:] < duration - 2)
references = {s: exact_counts(sim_spikes, edges, s) for s in (0.3, 0.05)}


def max_error(values, reference):
    error = np.abs(values[: reference.size] - reference)[interior]
    return error.max() / reference[interior].max()


print(f"{sim_st.n_events[0]} spikes; error = max |counts - exact| / max exact count\n")
print(f"{'pipeline':<38}{'sigma = 300 ms':>15}{'sigma = 50 ms':>15}")
for fine in (0.125, 0.025, 0.005, 0.001):
    w = round(target / fine)
    errors = [
        max_error(sim_st.bin(ds=fine).smooth(sigma=s).rebin(w=w).data[0], ref)
        for s, ref in references.items()
    ]
    label = f"bin {fine * 1000:g} ms -> smooth -> rebin(w={w})"
    print(f"{label:<38}" + "".join(f"{e:>15.2%}" for e in errors))
errors = [
    max_error(nel.utils.kernel_rate(sim_st, ds=target, sigma=s).data[0] * target, ref)
    for s, ref in references.items()
]
print(f"{'kernel_rate(ds=125 ms) x ds':<38}" + "".join(f"{e:>15.2%}" for e in errors))

With $\sigma$ = 300 ms, smoothing the 125 ms counts directly is already within 3% of the exact counts, and 25 ms fine bins bring it under 0.5%. With $\sigma$ = 50 ms, smoothing directly is 17% off, and the fine bins need to be about 5 ms ($\sigma$ / 10) to get under 1%. Finer is always more accurate, but 1 ms bins hold 125 times as many values as the 125 ms bins they are summed into.

`kernel_rate` multiplied by `ds` is accurate when the kernel is much wider than the bin, but not when it is narrower, for the same reason as in section 1.

## 3. Support edges and gaps

Here the spikes are dense just before a 200 ms gap between two support intervals, and there are no spikes after it.

In [ ]:
gap_st = nel.SpikeTrainArray(
    np.arange(0.8, 1.0, 0.01), support=nel.EpochArray([[0, 1], [1.2, 2.2]]), fs=1000
)
ds, sigma = 0.01, 0.1
default = gap_st.bin(ds=ds).smooth(sigma=sigma)
within = gap_st.bin(ds=ds).smooth(sigma=sigma, within_intervals=True)
kernel = nel.utils.kernel_rate(gap_st, ds=ds, sigma=sigma)
estimates = {
    "bin -> smooth() (default)": (default.bin_centers, default.data[0] / ds),
    "bin -> smooth(within_intervals=True)": (within.bin_centers, within.data[0] / ds),
    "kernel_rate": (kernel.abscissa_vals, kernel.data[0]),
}

fig, ax = plt.subplots(figsize=(10, 3.5))
for i, (name, (times, rate)) in enumerate(estimates.items()):
    for start, stop in gap_st.support.data:  # draw each interval separately
        inside = (times >= start) & (times < stop)
        ax.plot(
            times[inside],
            rate[inside],
            color=f"C{i}",
            ls=["-", "--", ":"][i],
            lw=2,
            label=name if start == 0 else None,
        )
ax.axvspan(1, 1.2, color="0.93", zorder=-1, label="gap in support")
ax.set(xlim=(0.5, 1.7), xlabel="time (s)", ylabel="rate (Hz)")
ax.legend(frameon=False)
plt.show()

for name, (times, rate) in estimates.items():
    after_gap = rate[np.argmin(np.abs(times - 1.205))]
    at_edge = rate[np.argmin(np.abs(times - 0.995))]
    print(
        f"{name:>37}: {at_edge:6.1f} Hz at the edge, {after_gap:5.1f} Hz just after the gap"
    )

By default, `smooth` treats the gap as missing data and smooths across it, so rate from the first interval leaks into the second, where no spikes were recorded. That is reasonable for a brief dropout but wrong when the intervals are separate epochs (for example, run sessions separated by rest). Pass `within_intervals=True`, or use `kernel_rate`, which never pools spikes across intervals.

At the edge itself the methods differ by design. `smooth` reflects the counts at the boundary, which assumes the rate continues as a mirror image; `kernel_rate` divides by the fraction of the kernel inside the support, which assumes the rate is locally constant.

## 4. Choosing $\sigma$ for each unit

`smooth` applies one $\sigma$ to every unit. `nel.utils.kernel_bandwidth` picks one per unit from its own spikes, by minimizing Shimazaki & Shinomoto's (2010) estimate of the squared error between the kernel rate and the unknown true rate (assuming Poisson spiking), and `kernel_rate` accepts one $\sigma$ per unit. Below, one simulated unit fires in brief 150 ms bursts and the other is slowly modulated over 15 s.

In [ ]:
def fast(t):
    return 2 + 30 * np.exp(-0.5 * (((t % 5) - 2.5) / 0.15) ** 2)


def slow(t):
    return 6 + 5 * np.sin(2 * np.pi * t / 15)


units = []
for rate_fn, peak in ((fast, 32), (slow, 11)):
    proposals = np.sort(rng.uniform(0, 60, rng.poisson(peak * 60)))
    units.append(proposals[rng.uniform(0, peak, proposals.size) < rate_fn(proposals)])
two_units = nel.SpikeTrainArray(units, support=nel.EpochArray([0, 60]), fs=30000)

sigmas = nel.utils.kernel_bandwidth(two_units)
own = nel.utils.kernel_rate(two_units, ds=0.01, sigma=sigmas)
swapped = nel.utils.kernel_rate(two_units, ds=0.01, sigma=sigmas[::-1])
t = own.abscissa_vals

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
for ax, rate_fn, name, i in zip(
    axes, (fast, slow), ("fast unit", "slow unit"), range(2)
):
    rmse_own = np.sqrt(np.mean((own.data[i] - rate_fn(t)) ** 2))
    rmse_swapped = np.sqrt(np.mean((swapped.data[i] - rate_fn(t)) ** 2))
    ax.plot(t, rate_fn(t), "k--", lw=1.2, label="true rate")
    ax.plot(
        t,
        own.data[i],
        lw=2,
        label=f"own sigma = {sigmas[i]:.2f} s (RMSE {rmse_own:.1f} Hz)",
    )
    ax.plot(
        t,
        swapped.data[i],
        lw=1,
        alpha=0.8,
        label=f"other unit's sigma = {sigmas[1 - i]:.2f} s (RMSE {rmse_swapped:.1f} Hz)",
    )
    ax.set(ylabel="rate (Hz)", title=name, xlim=(0, 30))
    ax.legend(
        frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1), fontsize="small"
    )
axes[-1].set(xlabel="time (s)")
plt.tight_layout()
plt.show()

`kernel_bandwidth` chose 0.15 s for the bursting unit and 1.11 s for the slowly modulated one. Swapping them shows why a single shared $\sigma$ is a compromise: the wide kernel flattens the bursts, and the narrow one turns the slow unit's rate into noise, roughly doubling to tripling the error (RMSE in the legends).

## Which pipeline to use

- **A firing rate over time** (plotting, correlating with behavior): `nel.utils.kernel_rate(st, ds=..., sigma=nel.utils.kernel_bandwidth(st))`.
- **Counts per decoding bin** (for `TuningCurve1D` or the decoders, which take a `BinnedSpikeTrainArray`): bin at a fine width that divides the decoding bin and is no more than about $\sigma / 10$, smooth, then `rebin` with a whole-number `w`.
- **Smoothing the decoding bins directly** is only accurate when the bin is much narrower than $\sigma$.
- **Separate epochs**: pass `within_intervals=True` to `smooth`, or use `kernel_rate`.
- **Not** `locsmooth` or `savgol_filter`: they are for continuous signals and can make spike rates negative.